In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm

from loka.data import load_breakthrough_states
from loka.models import BreakthroughCNN, AutoEncoder
from loka.geometrical_methods import herding_order, k_center_order
from loka.embeddings import with_counts, with_counts_and_near_goal, with_near_goal, counts, near_goal, clear_lane, _flat

import os
import pandas as pd

import torch

from cuml.manifold import TSNE

import numpy as np

from dotenv import load_dotenv
import torch.nn as nn

load_dotenv()

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

BASE_DIR = os.getenv("BASE_DIR")
X_flat,Y = load_breakthrough_states(f"{BASE_DIR}/data/game_states/e.txt", two_channel=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
def tsne_fig(X,Y, perplexity=30, random_seed=None, title=None):
    emb = TSNE(n_components=2, perplexity=perplexity, random_state=random_seed).fit_transform(X)  # numpy in, numpy out
    fig, ax = plt.subplots(figsize=(8, 8))
    for label, color in [(1, "tab:green"), (-1, "tab:red")]:
        m = Y == label
        ax.scatter(emb[m, 0], emb[m, 1], s=1, alpha=0.3, c=color, label=f"Y = {label}", rasterized=True)
    ax.legend(markerscale=10)

    if title is None:
        ax.set_title(f"t-SNE of {len(X):,} Breakthrough states")
    else:
        ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    return fig

In [ ]:
N = len(X_flat)
linear_tsne = tsne_fig(X_flat,Y, random_seed=SEED, title=f"t-SNE of {N:,} Breakthrough states\n(Only black and white positions)");

In [ ]:
encoder = nn.Sequential(
    nn.Linear(50, 35), nn.ReLU(),
    nn.Linear(35, 20), nn.ReLU(),
    nn.Linear(20, 10), nn.ReLU(),
)
decoder = nn.Sequential(
    nn.Linear(10, 20), nn.ReLU(),
    nn.Linear(20, 35), nn.ReLU(),
    nn.Linear(35, 50),
)
autoencoder = AutoEncoder(encoder, decoder).to(device)

X = torch.Tensor(X_flat).to(device)

optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

with tqdm(total=100, desc="Training") as pbar:
    autoencoder.train_model(X, optimizer, loss_fn, epochs=100, verbose=True, pbar=pbar)

In [ ]:
encoder = autoencoder.extract_encoder()

In [ ]:
enc_vals=encoder(X).detach().cpu()
enc_vals

In [ ]:
N = len(enc_vals)
linear_tsne = tsne_fig(enc_vals.numpy(),Y, random_seed=SEED, title=f"t-SNE of {N:,} Breakthrough states\n(Only black and white positions)");

In [ ]:
encoder = nn.Sequential(
    nn.Unflatten(1, (2, 5, 5)),                      # (N, 50) -> (N, 2, 5, 5)
    nn.Conv2d(2, 16, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
    nn.Flatten(),                                    # (N, 32*5*5)
    nn.Linear(32 * 5 * 5, 10), nn.ReLU(),            # bottleneck: 10
)
decoder = nn.Sequential(
    nn.Linear(10, 32 * 5 * 5), nn.ReLU(),
    nn.Unflatten(1, (32, 5, 5)),
    nn.Conv2d(32, 16, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(16, 2, kernel_size=3, padding=1),      # (N, 2, 5, 5)
    nn.Flatten(),                                    # (N, 50)
)
autoencoder = AutoEncoder(encoder, decoder).to(device)


X = torch.Tensor(X_flat).to(device)

optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

with tqdm(total=50, desc="Training") as pbar:
    autoencoder.train_model(X, optimizer, loss_fn, epochs=50, verbose=True, pbar=pbar)

In [ ]:
encoder = autoencoder.extract_encoder()
enc_vals=encoder(X).detach().cpu()
enc_vals

In [ ]:
autoencoder.eval()
with torch.no_grad():
    loss = loss_fn(autoencoder(X), X)
    print(loss)

In [ ]:
N = len(enc_vals)
linear_tsne = tsne_fig(enc_vals.numpy(),Y, random_seed=SEED, title=f"t-SNE of {N:,} Breakthrough states\n(Only black and white positions)");

# Embeddings


In [ ]:
class ToPlanes(nn.Module):
    """(N, 50 + k) -> (N, 2 + k, 5, 5): board as 2 channels, each extra field as a constant plane."""
    def forward(self, x):
        board = x[:, :50].reshape(-1, 2, 5, 5)
        extra = x[:, 50:, None, None].expand(-1, -1, 5, 5)
        return torch.cat([board, extra], dim=1)


class FromPlanes(nn.Module):
    """(N, 2 + k, 5, 5) -> (N, 50 + k): board channels flattened, each extra plane averaged to one value."""
    def forward(self, x):
        board = x[:, :2].flatten(1)
        extra = x[:, 2:].mean(dim=(2, 3))
        return torch.cat([board, extra], dim=1)


X_near_goal = with_near_goal(X_flat)   # (N, 52)
print(X_near_goal.shape)
X_near_goal=torch.Tensor(X_near_goal).to(device)
C = 4                              

encoder = nn.Sequential(
    ToPlanes(),                                      # (N, 52) -> (N, 4, 5, 5)
    nn.Conv2d(C, 16, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
    nn.Flatten(),                                    # (N, 32*5*5)
    nn.Linear(32 * 5 * 5, 10), nn.ReLU(),            # bottleneck: 10
)
decoder = nn.Sequential(
    nn.Linear(10, 32 * 5 * 5), nn.ReLU(),
    nn.Unflatten(1, (32, 5, 5)),
    nn.Conv2d(32, 16, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(16, C, kernel_size=3, padding=1),      # (N, 4, 5, 5)
    FromPlanes(),                                    # (N, 52)
)
autoencoder = AutoEncoder(encoder, decoder).to(device)

optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

with tqdm(total=50, desc="Training") as pbar:
    autoencoder.train_model(X_near_goal, optimizer, loss_fn, epochs=50, verbose=True, pbar=pbar)

In [ ]:
encoder = autoencoder.extract_encoder()
enc_vals=encoder(X_near_goal).detach().cpu()

autoencoder.eval()
with torch.no_grad():
    loss = loss_fn(autoencoder(X_near_goal), X_near_goal)
    print(loss)

N = len(enc_vals)
linear_tsne = tsne_fig(enc_vals.numpy(),Y, random_seed=SEED, title=f"t-SNE of {N:,} Breakthrough states\n(Only black and white positions)");

In [ ]:
X_e = np.concatenate([
    X_flat.reshape(len(X_flat), -1),   # (N, 50) positions
    counts(X_flat),                    # +2
    near_goal(X_flat),                 # +2
    clear_lane(X_flat),                # +2
], axis=1)
print(X_e.shape)                       # (N, 56)

X_ed = torch.Tensor(X_e).to(device)
C = 8 

In [ ]:
class ToPlanes(nn.Module):
    """(N, 50 + k) -> (N, 2 + k, 5, 5): board as 2 channels, each extra field as a constant plane."""
    def forward(self, x):
        board = x[:, :50].reshape(-1, 2, 5, 5)
        extra = x[:, 50:, None, None].expand(-1, -1, 5, 5)
        return torch.cat([board, extra], dim=1)


class FromPlanes(nn.Module):
    """(N, 2 + k, 5, 5) -> (N, 50 + k): board channels flattened, each extra plane averaged to one value."""
    def forward(self, x):
        board = x[:, :2].flatten(1)
        extra = x[:, 2:].mean(dim=(2, 3))
        return torch.cat([board, extra], dim=1)                          

encoder = nn.Sequential(
    ToPlanes(),                                      # (N, 52) -> (N, 4, 5, 5)
    nn.Conv2d(C, 16, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
    nn.Flatten(),                                    # (N, 32*5*5)
    nn.Linear(32 * 5 * 5, 10), nn.ReLU(),            # bottleneck: 10
)
decoder = nn.Sequential(
    nn.Linear(10, 32 * 5 * 5), nn.ReLU(),
    nn.Unflatten(1, (32, 5, 5)),
    nn.Conv2d(32, 16, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(16, C, kernel_size=3, padding=1),      # (N, 4, 5, 5)
    FromPlanes(),                                    # (N, 52)
)
autoencoder = AutoEncoder(encoder, decoder).to(device)

optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

with tqdm(total=50, desc="Training") as pbar:
    autoencoder.train_model(X_ed, optimizer, loss_fn, epochs=50, verbose=True, pbar=pbar)

In [ ]:
encoder = autoencoder.extract_encoder()
enc_vals=encoder(X_ed).detach().cpu()

autoencoder.eval()
with torch.no_grad():
    loss = loss_fn(autoencoder(X_ed), X_ed)
    print(loss)

N = len(enc_vals)
linear_tsne = tsne_fig(enc_vals.numpy(),Y, random_seed=SEED, title=f"t-SNE of {N:,} Breakthrough states\n(Only black and white positions)");